# Batch #5 - LLM attack on the concept confusion (Qwen2.5-3B-Instruct)

**Why this batch.** Four screens are now closed: recipe (batch 2), AUX head (batch 1-2),
off-diagonal emphasis (batch 4), threshold/prior calibration. Batch 4 was the informative one:
duplicating the off-diagonal quadrants *did* reduce the targeted FNs (up to -5.3 rows) but every
FN removed reappeared as +12 to +19 FPs on `no|no`. The decision boundary slides along one
confounded axis; reweighting cannot teach the concept.

Residual errors of the best ensemble (.7708, A160_x5+AUX_x5+E_cs2_x3), 160 total (81 FN / 79 FP):

| quadrant | n | errors | rate |
|---|---|---|---|
| yes\|no (polite stereotype) | 43 | 21 | **48.8%** |
| yes\|yes_implicit | 80 | 34 | **42.5%** |
| yes\|yes_explicit | 153 | 26 | 17.0% |
| no\|no | 300 | 39 | 13.0% |
| no\|yes_implicit | 95 | 18 | 18.9% |
| no\|yes_explicit (slur, no claim) | 67 | 22 | **32.8%** |

43/160 errors sit on the two off-diagonal quadrants. This is exactly "does the model know that a
*generalised claim* is a stereotype even when it is polite, and that a *slur* is not a stereotype
even when it is hateful". A 118M two-tower model trained on 7.6k noisy rows does not have that
world knowledge. An instruction-tuned LLM does, and it can also be *told* the annotation rule.

**Design.** Every stage uses one forward pass and a 2-way softmax over the `yes`/`no` token logits
at the answer position - no sampling, so all numbers are directly comparable and cheap:

| stage | what it measures |
|---|---|
| zero-shot | does the model already know the concept? |
| linear probe on frozen last-token features | how much of the concept is linearly decodable without any tuning? |
| few-shot (6 train examples) | can in-context examples fix the residual rule ambiguity? |
| LoRA (r=16, all linear) 2 epochs | how far does task fine-tuning get? |

Model: `Qwen/Qwen2.5-3B-Instruct` (ungated, multilingual EN/IT/NL, 6GB fp16, fits one T4 with
gradient checkpointing). Fallback to `Qwen/Qwen2.5-1.5B-Instruct` only on OOM. All stages are
independent and resume-safe: each writes its `.npy` before the next starts, and the notebook
re-zips after every stage.

**How to read it.** The bar is the local ensemble: .7708. A useful LLM result needs either
(a) LoRA >= .775 as a *single* model, or (b) a prob dump that adds to the ensemble locally
(we will test that offline). Zero-shot alone is already informative: if it is ~.70+ the concept is
in the weights and the ceiling is worth chasing; if it is ~.55 the labels are the wall.


In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/AmnO-O/Try.git'
PIN_COMMIT = '37c4c2645aea0dff1e749dc6f762f33d5e843d35'  # same pin as batches 1-4 (data/loader only)


def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)


if os.path.basename(os.path.abspath('.')) != 'Try':
    if not os.path.isdir('Try'):
        sh(f'git clone {REPO_URL}')
    os.chdir('Try')
if PIN_COMMIT:
    sh('git fetch origin main')
    sh(f'git checkout {PIN_COMMIT}')
sh('git log -1 --format="%H %cd %s" --date=iso')
sh('pip install -q -r requirements.txt')
sh('pip install -q peft')
print('cwd:', os.getcwd(), flush=True)


## 1. Data, split, and val arrays

Same split as every previous batch (`split_by_video`, val fraction 0.1, split seed 42) so the 738
val rows are row-for-row identical to the arrays the local analysis already holds.


In [ ]:
import glob
import json
import os
import sys
import time

hits = sorted(glob.glob('/kaggle/input/**/StereoQueerEval_*_training.tsv', recursive=True))
assert hits, 'StereoQueerEval TSVs not found - attach the dataset (Add Input > amnpham/lgbtqia)'
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs('weights', exist_ok=True)
print('data dir:', os.environ['LGBT_DATA_DIR'], flush=True)

import numpy as np
import pandas as pd
import torch

print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU T4 x2, then Run All'
print('gpu:', torch.cuda.get_device_name(0), flush=True)

sys.path.insert(0, os.path.abspath('src'))
import data as data_mod

df_all = data_mod.load_frame(languages=['EN', 'IT', 'NL'])
df_train, df_val = data_mod.split_by_video(df_all, 0.1, 42)
labs = df_val['st_y'].astype(int).to_numpy()
assert len(labs) == 738 and int(labs.sum()) == 276, \
    f'unexpected val frame: n={len(labs)} pos={int(labs.sum())}'
np.save('weights/val_labels.npy', labs)
np.save('weights/val_langs.npy', df_val['lang'].to_numpy(dtype=str))
np.save('weights/val_videos.npy', df_val['yt_title'].to_numpy(dtype=str))
quads = (df_val['stereotype'].astype(str) + '|' + df_val['hate_speech'].astype(str)).to_numpy(dtype=str)
np.save('weights/val_quads.npy', quads)
print('train', len(df_train), '| val', len(df_val), '| pos', int(labs.sum()), '| videos', df_val['yt_title'].nunique())
print('train by lang:', df_train['lang'].value_counts().to_dict())
print('val by quad:', pd.Series(quads).value_counts().to_dict())


## 2. Model + prompting

The system prompt encodes the annotation rule that the error analysis recovered from the labels
(polite generalisation = yes; slur/insult/counter-speech = no). The answer position is
`<|im_start|>assistant`, and we read the logits of the two answer tokens. Comments are truncated
before tokenisation (not after) so the assistant marker is never cut.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_CANDIDATES = ['Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct']
DEV = 'cuda:0'

SYS_PROMPT = (
    'You are an expert annotator for a multilingual stereotype-detection task about LGBTQ+ people.\n'
    '\n'
    'A comment CONTAINS A STEREOTYPE (answer yes) if it states, implies or endorses a generalised '
    'claim about LGBTQ+ people or a subgroup (gay men, lesbians, bisexual or transgender people). '
    'Examples: that they are inherently immoral, unnatural, dangerous, mentally ill or a threat; that '
    'all members of the group share a trait, behaviour or agenda; that they sexualise or recruit '
    'children; a generalisation disguised as a compliment, pity, a joke or a question. Politeness does '
    'not matter: a politely worded generalisation is still a stereotype.\n'
    '\n'
    'A comment does NOT contain a stereotype (answer no) if it is merely a slur, insult or expression '
    'of disgust aimed at a specific person or at LGBTQ+ people in general without a generalised claim; '
    'if it defends or supports LGBTQ+ people or argues against a stereotype; if it quotes scripture or '
    'gives a religious or personal opinion without generalising about the group; if it reports facts, '
    'news or a specific individual\'s behaviour; or if it attacks the video\'s speaker, opponents or '
    'other commenters rather than the group.\n'
    '\n'
    'Answer with exactly one word: yes or no.'
)

tok = AutoTokenizer.from_pretrained(MODEL_CANDIDATES[0])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'right'


def clean(s, n):
    return ' '.join(str(s).split())[:n]


def user_block(title, desc, comment, with_desc=True):
    parts = [f'Video title: {clean(title, 140)}']
    if with_desc:
        parts.append(f'Video description: {clean(desc, 320)}')
    parts.append(f'Comment: {clean(comment, 700)}')
    parts.append('Does this comment contain a stereotype about LGBTQ+ people?')
    return '\n'.join(parts)


def build_prompt(title, desc, comment, shots=()):
    msgs = [{'role': 'system', 'content': SYS_PROMPT}]
    for s in shots:
        msgs.append({'role': 'user', 'content': s[0]})
        msgs.append({'role': 'assistant', 'content': s[1]})
    msgs.append({'role': 'user', 'content': user_block(title, desc, comment)})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def answer_ids(prompt):
    p = tok(prompt, add_special_tokens=False).input_ids
    ids = {}
    for lab in ('yes', 'no'):
        f = tok(prompt + ' ' + lab, add_special_tokens=False).input_ids
        assert f[:len(p)] == p and len(f) == len(p) + 1, f'answer token is not a single token: {lab}'
        ids[lab] = f[-1]
    return ids


# Hand-picked in-context examples from the TRAIN split, covering the two hard quadrants
# (polite stereotype without hate = yes, slur/abuse without a claim = no) in all three languages.
SHOT_IDS = [
    ('training_EN_0157', 'yes'),  # yes|no  "very little love in the gay community"
    ('training_IT_0954', 'yes'),  # yes|no  rhetorical question about a trans child
    ('training_NL_0080', 'yes'),  # yes|no  "LGBT youth are all alternative/goth"
    ('training_EN_0148', 'yes'),  # yes|yes_implicit  "predatory agenda"
    ('training_EN_0300', 'no'),   # no|yes_explicit  insult/annoyance, no claim
    ('training_IT_0115', 'no'),   # no|yes_explicit  abuse at one person
    ('training_NL_0195', 'no'),   # no|yes_explicit  disgust, no claim
]
by_id = df_train.set_index('StereoQueerEval_id')
assert not set(i for i, _ in SHOT_IDS) & set(df_val['StereoQueerEval_id']), 'few-shot row leaked from val'
shots = []
for sid, name in SHOT_IDS:
    r = by_id.loc[sid]
    shots.append((user_block(r['yt_title'], r['yt_description'], r['yt_comment'], with_desc=False), name))
print('few-shot examples:')
for u, a in shots:
    print(' ', a, '|', u.replace(chr(10), ' / ')[:170])

probe_prompt = build_prompt(df_val['yt_title'].iloc[0], df_val['yt_description'].iloc[0],
                            df_val['yt_comment'].iloc[0])
ids0 = answer_ids(probe_prompt)
YES, NO = ids0['yes'], ids0['no']
print(f'yes id {YES} | no id {NO}')
print('zero-shot prompt tokens:', len(tok(probe_prompt, add_special_tokens=False).input_ids))
print('few-shot  prompt tokens:', len(tok(build_prompt(df_val['yt_title'].iloc[0], df_val['yt_description'].iloc[0],
                                                        df_val['yt_comment'].iloc[0], shots=shots),
                                        add_special_tokens=False).input_ids))

model = None
for name in MODEL_CANDIDATES:
    try:
        try:
            model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16,
                                                         attn_implementation='sdpa')
        except TypeError:
            model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16,
                                                         attn_implementation='sdpa')
        model.to(DEV)
        model.config.use_cache = False
        print('loaded', name, flush=True)
        break
    except torch.cuda.OutOfMemoryError:
        print('OOM loading', name, '- falling back', flush=True)
        torch.cuda.empty_cache()
        model = None
assert model is not None, 'no model could be loaded'
n_par = sum(p.numel() for p in model.parameters()) / 1e9
print(f'params {n_par:.2f}B | dtype {next(model.parameters()).dtype}')


## 3. One forward pass: zero-shot probs + frozen last-token features

The same pass gives (a) the yes/no logits and (b) the last-layer hidden state at the last token,
which the linear probe consumes. Nothing is sampled.


In [ ]:
import numpy as np
import torch

N = len(df_val)
L = model.config.num_hidden_layers + 1


@torch.no_grad()
def forward_pass(prompts, bs=6, want_hidden=False, tag=''):
    model.eval()
    probs = np.zeros(len(prompts), dtype=np.float32)
    hid = np.zeros((len(prompts), model.config.hidden_size), dtype=np.float16) if want_hidden else None
    for i in range(0, len(prompts), bs):
        chunk = prompts[i:i + bs]
        enc = tok(chunk, return_tensors='pt', padding=True, truncation=True, max_length=2048,
                  add_special_tokens=False)
        enc = {k: v.to(DEV) for k, v in enc.items()}
        out = model(**enc, output_hidden_states=want_hidden)
        last = enc['attention_mask'].sum(1) - 1
        row = torch.arange(len(chunk), device=DEV)
        two = out.logits[row, last][:, [YES, NO]].float()
        probs[i:i + len(chunk)] = torch.softmax(two, -1)[:, 0].cpu().numpy()
        if want_hidden:
            hid[i:i + len(chunk)] = out.hidden_states[-1][row, last].cpu().numpy().astype(np.float16)
        if i % 120 == 0:
            print(f'  {tag} {i}/{len(prompts)}', flush=True)
    return probs, hid


t0 = time.time()
zs_prompts = [build_prompt(t, d, c) for t, d, c in
              zip(df_val['yt_title'], df_val['yt_description'], df_val['yt_comment'])]
zs_probs, zs_hid = forward_pass(zs_prompts, bs=6, want_hidden=True, tag='zero-shot')
np.save('weights/llm_zeroshot_val_probs.npy', zs_probs)
np.save('weights/llm_val_last_hidden.npy', zs_hid)
print(f'zero-shot pass {time.time() - t0:.0f}s | mean p {zs_probs.mean():.3f} '
      f'| predicted pos {(zs_probs >= 0.5).mean():.3f}')


## 4. Frozen-LLM linear probe

Logistic regression on the last-token hidden state, C chosen by 3-fold CV **on train** (val labels
never used for selection). This measures how much of the concept is already linearly present.


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

y_tr = df_train['st_y'].astype(int).to_numpy()
tr_prompts = [build_prompt(t, d, c) for t, d, c in
              zip(df_train['yt_title'], df_train['yt_description'], df_train['yt_comment'])]
t0 = time.time()
_, tr_hid = forward_pass(tr_prompts, bs=6, want_hidden=True, tag='train-feats')
print(f'train feature pass {time.time() - t0:.0f}s', flush=True)
np.save('weights/llm_train_last_hidden.npy', tr_hid)

X = np.asarray(tr_hid, dtype=np.float32)
Xv = np.asarray(zs_hid, dtype=np.float32)
sc = StandardScaler().fit(X)
Xs, Xvs = sc.transform(X), sc.transform(Xv)

best_c, best_cv = None, -1.0
for C in (0.003, 0.01, 0.03, 0.1, 0.3):
    f1s = []
    for tr_i, va_i in StratifiedKFold(3, shuffle=True, random_state=0).split(Xs, y_tr):
        clf = LogisticRegression(C=C, max_iter=2000, n_jobs=-1).fit(Xs[tr_i], y_tr[tr_i])
        f1s.append(f1_score(y_tr[va_i], clf.predict(Xs[va_i]), average='macro'))
    cv = float(np.mean(f1s))
    print(f'  C={C:<6} train-CV macro F1 {cv:.4f}')
    if cv > best_cv:
        best_c, best_cv = C, cv
print(f'selected C={best_c} (train CV {best_cv:.4f})')

clf = LogisticRegression(C=best_c, max_iter=2000, n_jobs=-1).fit(Xs, y_tr)
probe_probs = clf.predict_proba(Xvs)[:, 1].astype(np.float32)
np.save('weights/llm_probe_val_probs.npy', probe_probs)
print(f'linear probe val: {f1_score(labs, (probe_probs >= 0.5).astype(int), average="macro"):.4f}')


## 5. Few-shot probe

Six in-context train examples (1 yes + 1 no per language, short comments, no descriptions to keep
the prompt small). Same answer-position readout.


In [ ]:
t0 = time.time()
fs_prompts = [build_prompt(t, d, c, shots=shots) for t, d, c in
              zip(df_val['yt_title'], df_val['yt_description'], df_val['yt_comment'])]
fs_probs, _ = forward_pass(fs_prompts, bs=2, tag='few-shot')
np.save('weights/llm_fewshot_val_probs.npy', fs_probs)
print(f'few-shot pass {time.time() - t0:.0f}s')


## 6. LoRA fine-tune

2-way CE on the yes/no logits at the answer position, LoRA r=16 on all linear layers, 2 epochs,
per-epoch val evaluation. Checkpointing on, batch 4 x grad-accum 4, max length 512. The adapter is
kept only if val macro F1 improves; probs from the best epoch are saved.


In [ ]:
import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset

MAXLEN = 768  # measured max prompt is 623 tokens; 512 would truncate the assistant marker


def tokenize_prompt(p):
    ids = tok(p, add_special_tokens=False, truncation=True, max_length=MAXLEN).input_ids
    return ids


train_ids = [tokenize_prompt(p) for p in tr_prompts]
val_ids = [tokenize_prompt(p) for p in zs_prompts]
print('max train tokens', max(len(x) for x in train_ids), '| max val', max(len(x) for x in val_ids))
assert max(len(x) for x in train_ids) < MAXLEN and max(len(x) for x in val_ids) < MAXLEN, \
    'prompt hit MAXLEN - the assistant marker would be truncated'


class DS(Dataset):
    def __init__(self, ids, ys):
        self.ids, self.ys = ids, ys

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i], int(self.ys[i])


PAD = tok.pad_token_id


def collate(batch):
    m = max(len(x[0]) for x in batch)
    inp = torch.full((len(batch), m), PAD, dtype=torch.long)
    att = torch.zeros((len(batch), m), dtype=torch.long)
    ys = torch.zeros(len(batch), dtype=torch.long)
    for i, (ids, y) in enumerate(batch):
        inp[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        att[i, :len(ids)] = 1
        ys[i] = y
    return inp, att, ys


@torch.no_grad()
def eval_ids(ids_list, bs=8):
    model.eval()
    out = np.zeros(len(ids_list), dtype=np.float32)
    for i in range(0, len(ids_list), bs):
        batch = [(x, 0) for x in ids_list[i:i + bs]]
        inp, att, _ = collate(batch)
        logits = model(input_ids=inp.to(DEV), attention_mask=att.to(DEV)).logits
        last = att.sum(1) - 1
        row = torch.arange(len(batch), device=DEV)
        two = logits[row, last.to(DEV)][:, [YES, NO]].float()
        out[i:i + len(batch)] = torch.softmax(two, -1)[:, 0].cpu().numpy()
    return out


lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                  'gate_proj', 'up_proj', 'down_proj'],
                  task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False
model.print_trainable_parameters()

EPOCHS, BS, ACCUM, LR = 2, 4, 4, 2e-4
loader = DataLoader(DS(train_ids, y_tr), batch_size=BS, shuffle=True, collate_fn=collate,
                    num_workers=0, drop_last=True)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
steps_total = (len(loader) // ACCUM) * EPOCHS
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps_total,
                                            pct_start=0.05, anneal_strategy='cos')
try:
    scaler = torch.amp.GradScaler('cuda', enabled=True)
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=True)
print(f'steps/epoch {len(loader)} | optimiser steps {steps_total}', flush=True)

best_f1, best_probs, hist = -1.0, None, []
t0 = time.time()
for ep in range(EPOCHS):
    model.train()
    run, nb = 0.0, 0
    for step, (inp, att, ys) in enumerate(loader):
        out = model(input_ids=inp.to(DEV), attention_mask=att.to(DEV))
        last = (att.sum(1) - 1).to(DEV)
        row = torch.arange(len(ys), device=DEV)
        two = out.logits[row, last][:, [YES, NO]].float()
        loss = F.cross_entropy(two, ys.to(DEV))
        scaler.scale(loss / ACCUM).backward()
        run += float(loss)
        nb += 1
        if (step + 1) % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad(set_to_none=True)
        if step % 200 == 0:
            print(f'  ep{ep + 1} step {step}/{len(loader)} loss {run / max(nb, 1):.4f} '
                  f'({time.time() - t0:.0f}s)', flush=True)
            run, nb = 0.0, 0
    opt.zero_grad(set_to_none=True)  # drop the tail of the last accumulation window
    vp = eval_ids(val_ids)
    f1 = f1_score(labs, (vp >= 0.5).astype(int), average='macro')
    hist.append({'epoch': ep + 1, 'val_macro_f1': float(f1), 'seconds': round(time.time() - t0, 1)})
    print(f'== epoch {ep + 1}: val macro F1 {f1:.4f}', flush=True)
    if f1 > best_f1:
        best_f1, best_probs = f1, vp
        model.save_pretrained('weights/llm_lora_best')
        print('   saved new best adapter', flush=True)

np.save('weights/llm_lora_val_probs.npy', best_probs)
with open('weights/llm_lora_history.json', 'w') as fh:
    json.dump({'hist': hist, 'best_f1': best_f1, 'model': MODEL_CANDIDATES[0],
               'epochs': EPOCHS, 'bs': BS, 'accum': ACCUM, 'lr': LR, 'lora_r': 16}, fh, indent=1)
print(f'best LoRA val macro F1 {best_f1:.4f} | total {time.time() - t0:.0f}s')


## 7. Summary + dumps

Prints every stage against the local ensemble bar (.7708), broken down by quadrant and language, and
writes the raw probs plus a CSV for local analysis.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

BAR = 0.7708
stage_probs = {
    'llm_zeroshot': np.load('weights/llm_zeroshot_val_probs.npy'),
    'llm_fewshot': np.load('weights/llm_fewshot_val_probs.npy'),
    'llm_probe': np.load('weights/llm_probe_val_probs.npy'),
    'llm_lora': np.load('weights/llm_lora_val_probs.npy'),
}
langs = np.load('weights/val_langs.npy').astype(str)
quads = np.load('weights/val_quads.npy').astype(str)

print(f'{"stage":14} {"macroF1":>8} {"acc":>7} {"pos_rate":>9}   per-language F1')
print('-' * 78)
for name, p in stage_probs.items():
    pred = (p >= 0.5).astype(int)
    f1 = f1_score(labs, pred, average='macro')
    per = ' '.join(f'{lg} {f1_score(labs[langs == lg], pred[langs == lg], average="macro"):.3f}'
                   for lg in ('EN', 'IT', 'NL'))
    print(f'{name:14} {f1:8.4f} {(pred == labs).mean():7.4f} {pred.mean():9.3f}   {per}')

print()
print('quadrant errors (FN for yes|*, FP for no|*)')
print(f'{"quad":16} {"n":>4} ' + ' '.join(f'{n.replace("llm_", ""):>10}' for n in stage_probs))
for q in ('yes|no', 'yes|yes_implicit', 'yes|yes_explicit', 'no|no', 'no|yes_implicit', 'no|yes_explicit'):
    sel = quads == q
    cells = []
    for p in stage_probs.values():
        pred = (p[sel] >= 0.5).astype(int)
        err = int((pred == 0).sum()) if q.startswith('yes') else int((pred == 1).sum())
        cells.append(f'{err:>4}/{sel.sum():<5}')
    print(f'{q:16} {sel.sum():>4} ' + ' '.join(f'{c:>10}' for c in cells))

pred_df = pd.DataFrame({'id': df_val['StereoQueerEval_id'].to_numpy(),
                        'lang': langs, 'quad': quads, 'gold': labs})
for name, p in stage_probs.items():
    pred_df[f'{name}_p'] = p
pred_df.to_csv('weights/llm_val_predictions.csv', index=False)
summary = {name: float(f1_score(labs, (p >= 0.5).astype(int), average='macro'))
           for name, p in stage_probs.items()}
with open('weights/llm_summary.json', 'w') as fh:
    json.dump({'summary': summary, 'bar_local_ensemble': BAR,
               'lora_best_by_val': 'see llm_lora_history.json'}, fh, indent=1)
print()
print('summary json:', json.dumps(summary, indent=1))

import shutil
shutil.make_archive('/kaggle/working/llm_results', 'zip', 'weights')
print('zip:', os.path.getsize('/kaggle/working/llm_results.zip') / 1e6, 'MB')
